In [3]:
import numpy as np, pandas as pd, lightgbm as lgb
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [4]:
CLASES = ["SLSN-I", "SN II", "SN IIn", "SN Ia", "SN Ib/c"]
A = pd.read_csv("../data/features.csv").sort_values("NAME").reset_index(drop=True)
B = pd.read_csv("../data/features_nested.csv").sort_values("NAME").reset_index(drop=True)
assert (A["NAME"] == B["NAME"]).all()
y = A["CLASE"].map({c: i for i, c in enumerate(CLASES)}).to_numpy()

def cv(Xf, seed):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    pred = np.empty(len(y), dtype=int)
    for tr, te in skf.split(Xf, y):
        m = lgb.LGBMClassifier(boosting_type="dart", n_estimators=250, max_depth=5,
                               num_leaves=20, reg_lambda=5, class_weight="balanced",
                               random_state=seed, verbose=-1).fit(Xf[tr], y[tr])
        pred[te] = m.predict(Xf[te])
    return pred

filas = []
for nombre, D in [("curve_fit (10 feat.)", A), ("nested (12 feat.)", B)]:
    Xf = D[[c for c in D.columns if c not in ("NAME", "CLASE")]].to_numpy()
    for s in range(5):
        p = cv(Xf, s)
        cm = confusion_matrix(y, p)
        filas.append({"features": nombre, "semilla": s,
                      "accuracy": accuracy_score(y, p),
                      "F1_macro": f1_score(y, p, average="macro"),
                      **{f"comp_{c}": v for c, v in zip(CLASES, cm.diagonal() / cm.sum(axis=1))},
                      **{f"pur_{c}": v for c, v in zip(CLASES, cm.diagonal() / cm.sum(axis=0))}})

res_f = pd.DataFrame(filas)
res_f.to_csv("../results/comparacion_features.csv", index=False)
print(res_f.drop(columns="semilla").groupby("features").agg(["mean", "std"]).T.round(3).to_string())

features           curve_fit (10 feat.)  nested (12 feat.)
accuracy     mean                 0.828              0.858
             std                  0.004              0.004
F1_macro     mean                 0.540              0.622
             std                  0.007              0.003
comp_SLSN-I  mean                 0.284              0.487
             std                  0.072              0.038
comp_SN II   mean                 0.718              0.753
             std                  0.007              0.013
comp_SN IIn  mean                 0.509              0.580
             std                  0.012              0.019
comp_SN Ia   mean                 0.892              0.912
             std                  0.006              0.004
comp_SN Ib/c mean                 0.531              0.642
             std                  0.011              0.024
pur_SLSN-I   mean                 0.200              0.400
             std                  0.028              0.0

El método de ajuste importa. Pasar de mínimos cuadrados a nested samplin hace subir el F1 macro de 0,54 a 0,62, y la mayor mejora está en las clases raras. Esto da contenido a tu análisis de discrepancias.
En este experimento cambiaron dos cosas a la vez, el método de ajuste y que ahora se tiene sigma_extra 12 features en vez de 10. Es posible que σ_extra por sí solo explique buena parte, porque mide cuánto se aparta la curva del modelo, y eso puede distinguir clases con curvas irregulares.
Este F1 es el de las predicciones juntas de todos los folds, y el del paper es la mediana de los folds.

In [5]:
def corre(nombre, D):
    Xf = D[[c for c in D.columns if c not in ("NAME", "CLASE")]].to_numpy()
    out = []
    for s in range(5):
        p = cv(Xf, s); cm = confusion_matrix(y, p)
        out.append({"features": nombre, "semilla": s,
                    "accuracy": accuracy_score(y, p),
                    "F1_macro": f1_score(y, p, average="macro"),
                    **{f"comp_{c}": v for c, v in zip(CLASES, cm.diagonal() / cm.sum(axis=1))},
                    **{f"pur_{c}": v for c, v in zip(CLASES, cm.diagonal() / cm.sum(axis=0))}})
    return out

res_abl = pd.DataFrame(corre("nested sin sigma_extra (10 feat.)",
                             B.drop(columns=["sigma_extra_r", "sigma_extra_g"])))
res_all = pd.concat([res_f, res_abl])
res_all.to_csv("../results/comparacion_features.csv", index=False)
cols = ["accuracy", "F1_macro", "comp_SLSN-I", "comp_SN Ib/c", "pur_SLSN-I"]
print(res_all.groupby("features")[cols].mean().round(3))

                                   accuracy  F1_macro  comp_SLSN-I  \
features                                                             
curve_fit (10 feat.)                  0.828     0.540        0.284   
nested (12 feat.)                     0.858     0.622        0.487   
nested sin sigma_extra (10 feat.)     0.854     0.618        0.499   

                                   comp_SN Ib/c  pur_SLSN-I  
features                                                     
curve_fit (10 feat.)                      0.531       0.200  
nested (12 feat.)                         0.642       0.400  
nested sin sigma_extra (10 feat.)         0.647       0.386  


In [6]:
p = res_f.pivot(index="semilla", columns="features", values="F1_macro")
print((p["nested (12 feat.)"] > p["curve_fit (10 feat.)"]).sum(), "de 5 semillas")

5 de 5 semillas


In [7]:
SEED = 42
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
Xf = B[[c for c in B.columns if c not in ("NAME", "CLASE")]].to_numpy()

pred = np.empty(len(y), dtype=int); acc, f1 = [], []
for tr, te in skf.split(Xf, y):
    m = lgb.LGBMClassifier(boosting_type="dart", n_estimators=250, max_depth=5, num_leaves=20,
                           reg_lambda=5, class_weight="balanced", random_state=SEED, verbose=-1).fit(Xf[tr], y[tr])
    pred[te] = m.predict(Xf[te])
    acc.append(accuracy_score(y[te], pred[te]))
    f1.append(f1_score(y[te], pred[te], average="macro"))

def resumen(v):
    v = np.sort(v)
    return round(float(np.median(v)), 3), round(float(v[1]), 3), round(float(v[-2]), 3)

print("accuracy (mediana, 2º menor, 2º mayor):", resumen(acc))
print("F1 macro (mediana, 2º menor, 2º mayor):", resumen(f1))

cm = confusion_matrix(y, pred)
tabla = pd.DataFrame({
    "comp_paper": [0.75, 0.72, 0.52, 0.87, 0.80], "comp_nested": (cm.diagonal() / cm.sum(axis=1)).round(2),
    "pur_paper":  [0.27, 0.87, 0.51, 0.97, 0.28], "pur_nested":  (cm.diagonal() / cm.sum(axis=0)).round(2)},
    index=CLASES)
print(tabla)
tabla.to_csv("../results/comparacion_paper_vs_nested.csv")

accuracy (mediana, 2º menor, 2º mayor): (0.847, 0.835, 0.863)
F1 macro (mediana, 2º menor, 2º mayor): (0.593, 0.571, 0.652)
         comp_paper  comp_nested  pur_paper  pur_nested
SLSN-I         0.75         0.48       0.27        0.38
SN II          0.72         0.72       0.87        0.77
SN IIn         0.52         0.53       0.51        0.46
SN Ia          0.87         0.91       0.97        0.96
SN Ib/c        0.80         0.66       0.28        0.31
